In [9]:
import sys

sys.path.append("../src")

from pathlib import Path

import numpy as np
import pandas as pd

from energy_load_forecast.ingestion import load_and_resample
from energy_load_forecast.preprocessing import (
    add_calendar_features,
    chronological_split,
    clean_tail,
)

In [10]:
series = load_and_resample(Path("../data/raw/entsoe/load_RO_2026-09-13_2026-09-20.csv"))
series_clean = clean_tail(series)
df = add_calendar_features(series_clean.to_frame())

train, val, test = chronological_split(df, "2026-09-16", "2026-09-18")
print("Shapes (train, val, test):", train.shape, val.shape, test.shape)

Shapes (train, val, test): (72, 5) (48, 5) (23, 5)


In [3]:
history = pd.concat([train["load_mw"], val["load_mw"]])
horizon = len(test)
print("History length:", len(history))
print("Horizon:", horizon)

History length: 120
Horizon: 23


In [4]:
last_value = history.iloc[-1]
future_index = pd.date_range(history.index[-1] + pd.Timedelta(hours=1), periods=horizon, freq="h", tz="UTC")
naive_pred = pd.Series(last_value, index=future_index, name="forecast")
print("Naive forecast:\n", naive_pred.head())

Naive forecast:
 2026-09-18 00:00:00+00:00    4840.25
2026-09-18 01:00:00+00:00    4840.25
2026-09-18 02:00:00+00:00    4840.25
2026-09-18 03:00:00+00:00    4840.25
2026-09-18 04:00:00+00:00    4840.25
Freq: h, Name: forecast, dtype: float64


In [5]:
season_length = 24
seasonal_block = history.iloc[-season_length:]
n_tiles = -(-horizon // season_length)
values = list(seasonal_block.to_numpy()) * n_tiles
seasonal_pred = pd.Series(values[:horizon], index=future_index, name="forecast")
print("Seasonal naive forecast:\n", seasonal_pred.head())

Seasonal naive forecast:
 2026-09-18 00:00:00+00:00    5083.25
2026-09-18 01:00:00+00:00    5192.25
2026-09-18 02:00:00+00:00    5263.25
2026-09-18 03:00:00+00:00    5535.00
2026-09-18 04:00:00+00:00    5989.50
Freq: h, Name: forecast, dtype: float64


In [6]:
expected_first = history.iloc[-24]
actual_first = seasonal_pred.iloc[0]
print("Expected:", expected_first, "Actual:", actual_first, "Match:", expected_first == actual_first)

Expected: 5083.25 Actual: 5083.25 Match: True


In [7]:
actual = test["load_mw"].iloc[:horizon]
actual.index = naive_pred.index


def mae(a, p):
    return float(np.mean(np.abs(a.to_numpy() - p.to_numpy())))


def rmse(a, p):
    return float(np.sqrt(np.mean((a.to_numpy() - p.to_numpy()) ** 2)))


def mape(a, p):
    return float(np.mean(np.abs((a.to_numpy() - p.to_numpy()) / a.to_numpy())) * 100)

In [8]:
results = pd.DataFrame(
    [
        {"MAE": mae(actual, naive_pred), "RMSE": rmse(actual, naive_pred), "MAPE": mape(actual, naive_pred)},
        {"MAE": mae(actual, seasonal_pred), "RMSE": rmse(actual, seasonal_pred), "MAPE": mape(actual, seasonal_pred)},
    ],
    index=["Naive", "Seasonal Naive"],
)
print("Results:\n", results)

Results:
                       MAE        RMSE       MAPE
Naive           645.73913  865.271671  10.854193
Seasonal Naive  235.01087  313.850036   4.581116
